# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [22]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
Path("models").mkdir(exist_ok=True)

In [4]:
df = pd.read_csv("data/BankChurners.csv")
df = df.iloc[:, :-2] #Recommendation from the creator of the Kaggle dataset 

display(df.head())
display(df.describe())
display(df.shape)

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


,CLIENTNUM,Customer_Age,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
count,1.012700e+04,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000
mean,7.391776e+08,46.325960,2.346203,35.928409,3.812580,2.341167,2.455317,8631.953698,1162.814061,7469.139637,0.759941,4404.086304,64.858695,0.712222,0.274894
std,3.690378e+07,8.016814,1.298908,7.986416,1.554408,1.010622,1.106225,9088.776650,814.987335,9090.685324,0.219207,3397.129254,23.472570,0.238086,0.275691
min,7.080821e+08,26.000000,0.000000,13.000000,1.000000,0.000000,0.000000,1438.300000,0.000000,3.000000,0.000000,510.000000,10.000000,0.000000,0.000000
25%,7.130368e+08,41.000000,1.000000,31.000000,3.000000,2.000000,2.000000,2555.000000,359.000000,1324.500000,0.631000,2155.500000,45.000000,0.582000,0.023000
50%,7.179264e+08,46.000000,2.000000,36.000000,4.000000,2.000000,2.000000,4549.000000,1276.000000,3474.000000,0.736000,3899.000000,67.000000,0.702000,0.176000
75%,7.731435e+08,52.000000,3.000000,40.000000,5.000000,3.000000,3.000000,11067.500000,1784.000000,9859.000000,0.859000,4741.000000,81.000000,0.818000,0.503000
max,8.283431e+08,73.000000,5.000000,56.000000,6.000000,6.000000,6.000000,34516.000000,2517.000000,34516.000000,3.397000,18484.000000,139.000000,3.714000,0.999000


(10127, 21)

In [5]:
print(df.dtypes)

CLIENTNUM                     int64
Attrition_Flag                  str
Customer_Age                  int64
Gender                          str
Dependent_count               int64
Education_Level                 str
Marital_Status                  str
Income_Category                 str
Card_Category                   str
Months_on_book                int64
Total_Relationship_Count      int64
Months_Inactive_12_mon        int64
Contacts_Count_12_mon         int64
Credit_Limit                float64
Total_Revolving_Bal           int64
Avg_Open_To_Buy             float64
Total_Amt_Chng_Q4_Q1        float64
Total_Trans_Amt               int64
Total_Trans_Ct                int64
Total_Ct_Chng_Q4_Q1         float64
Avg_Utilization_Ratio       float64
dtype: object


In [6]:
y = (df["Attrition_Flag"] == "Attrited Customer").astype(int)

print(y.value_counts())

Attrition_Flag
0    8500
1    1627
Name: count, dtype: int64


In [7]:
print("Total missing values:", df.isna().sum().sum())

text_cols = df.select_dtypes(include="object").columns.tolist()
for col in text_cols:
    print(col, "->", df[col].unique())

Total missing values: 0
Attrition_Flag -> <StringArray>
['Existing Customer', 'Attrited Customer']
Length: 2, dtype: str
Gender -> <StringArray>
['M', 'F']
Length: 2, dtype: str
Education_Level -> <StringArray>
[  'High School',      'Graduate',    'Uneducated',       'Unknown',
       'College', 'Post-Graduate',     'Doctorate']
Length: 7, dtype: str
Marital_Status -> <StringArray>
['Married', 'Single', 'Unknown', 'Divorced']
Length: 4, dtype: str
Income_Category -> <StringArray>
[   '$60K - $80K', 'Less than $40K',   '$80K - $120K',    '$40K - $60K',
        '$120K +',        'Unknown']
Length: 6, dtype: str
Card_Category -> <StringArray>
['Blue', 'Gold', 'Silver', 'Platinum']
Length: 4, dtype: str


/tmp/ipykernel_24526/1391275572.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns.tolist()


In [8]:
num_cols = df.select_dtypes(include="number").columns
corr = df[num_cols].corrwith(y).sort_values()
print(corr)

Total_Trans_Ct             -0.371403
Total_Ct_Chng_Q4_Q1        -0.290054
Total_Revolving_Bal        -0.263053
Avg_Utilization_Ratio      -0.178410
Total_Trans_Amt            -0.168598
Total_Relationship_Count   -0.150005
Total_Amt_Chng_Q4_Q1       -0.131063
CLIENTNUM                  -0.046430
Credit_Limit               -0.023873
Avg_Open_To_Buy            -0.000285
Months_on_book              0.013687
Customer_Age                0.018203
Dependent_count             0.018991
Months_Inactive_12_mon      0.152449
Contacts_Count_12_mon       0.204491
dtype: float64


In [9]:
cat_cols = ["Gender", "Education_Level", "Marital_Status", "Income_Category", "Card_Category"]

for col in cat_cols:
    print(y.groupby(df[col]).mean())
    print()

Gender
F    0.173572
M    0.146152
Name: Attrition_Flag, dtype: float64

Education_Level
College          0.152024
Doctorate        0.210643
Graduate         0.155691
High School      0.152012
Post-Graduate    0.178295
Uneducated       0.159381
Unknown          0.168532
Name: Attrition_Flag, dtype: float64

Marital_Status
Divorced    0.161765
Married     0.151269
Single      0.169414
Unknown     0.172230
Name: Attrition_Flag, dtype: float64

Income_Category
$120K +           0.173315
$40K - $60K       0.151397
$60K - $80K       0.134807
$80K - $120K      0.157655
Less than $40K    0.171862
Unknown           0.168165
Name: Attrition_Flag, dtype: float64

Card_Category
Blue        0.160979
Gold        0.181034
Platinum    0.250000
Silver      0.147748
Name: Attrition_Flag, dtype: float64



In [11]:
keep_cols = [
    "Customer_Age", "Income_Category", "Months_on_book", "Total_Relationship_Count",
    "Months_Inactive_12_mon", "Contacts_Count_12_mon", "Credit_Limit",
    "Total_Revolving_Bal", "Avg_Utilization_Ratio", "Total_Trans_Amt",
    "Total_Trans_Ct", "Total_Ct_Chng_Q4_Q1",
]

X = df[keep_cols].copy()
X = pd.get_dummies(X, columns=["Income_Category"], drop_first=True).astype(float)

print(X.shape)
print(X.columns.tolist())

(10127, 16)
['Customer_Age', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Utilization_Ratio', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Income_Category_$40K - $60K', 'Income_Category_$60K - $80K', 'Income_Category_$80K - $120K', 'Income_Category_Less than $40K', 'Income_Category_Unknown']


In [12]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)

print("train:", X_train.shape, "churn rate:", y_train.mean().round(3))
print("val:  ", X_val.shape, "churn rate:", y_val.mean().round(3))
print("test: ", X_test.shape, "churn rate:", y_test.mean().round(3))

train: (7088, 16) churn rate: 0.161
val:   (1519, 16) churn rate: 0.161
test:  (1520, 16) churn rate: 0.161


In [13]:
scaler = StandardScaler().fit(X_train)

X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

def score(y_true, proba, threshold=0.50):
    pred = (proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, proba),
    }

# Baseline
prob_base = np.zeros(len(y_test))
print(score(y_test, prob_base))

{'accuracy': 0.8394736842105263, 'precision': 0.0, 'recall': 0.0, 'roc_auc': 0.5}


In [15]:
# Model 1 : Logistic Regression

from sklearn.linear_model import LogisticRegression

C = 0.5
classifier = LogisticRegression(C=C, max_iter=1000)
classifier.fit(X_train_s, y_train)

prob_sk = classifier.predict_proba(X_test_s)[:, 1]
print(score(y_test, prob_sk))

{'accuracy': 0.9, 'precision': 0.7738095238095238, 'recall': 0.5327868852459017, 'roc_auc': 0.9092868338557994}


In [16]:
# Model 2 : Manual PyTorchLoop

Xt = torch.tensor(X_train_s, dtype=torch.float32)
yt = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
Xte = torch.tensor(X_test_s, dtype=torch.float32)

LR = 0.5
EPOCHS = 10000
lam = 1 / (2 * C * len(X_train))   # makes the penalty match sklearn's C

w = torch.zeros(Xt.shape[1], requires_grad=True)
b = torch.zeros(1, requires_grad=True)

for epoch in range(EPOCHS):
    p = torch.sigmoid(Xt @ w + b)
    loss = -(yt * torch.log(p + 1e-8) + (1 - yt) * torch.log(1 - p + 1e-8)).mean() + lam * (w ** 2).sum()
    loss.backward()
    with torch.no_grad():
        w -= LR * w.grad
        b -= LR * b.grad
    w.grad.zero_()
    b.grad.zero_()
    if epoch % 500 == 0:
        print(epoch, round(loss.item(), 4))

prob_manual = torch.sigmoid(Xte @ w + b).detach().numpy()
print(pd.Series(score(y_test, prob_manual)).round(3))

0 0.6931
500 0.2371
1000 0.2368
1500 0.2368
2000 0.2368
2500 0.2368
3000 0.2368
3500 0.2368
4000 0.2368
4500 0.2368
5000 0.2368
5500 0.2368
6000 0.2368
6500 0.2368
7000 0.2368
7500 0.2368
8000 0.2368
8500 0.2368
9000 0.2368
9500 0.2368
accuracy     0.900
precision    0.774
recall       0.533
roc_auc      0.909
dtype: float64


In [17]:
# Model 3 :  Standar PyTorch workflow
class LogReg(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.linear(x)).squeeze(-1)

model = LogReg(Xt.shape[1])
nn.init.zeros_(model.linear.weight)
nn.init.zeros_(model.linear.bias)

optimizer = torch.optim.SGD(model.parameters(), lr=LR)
loss_fn = nn.BCELoss()

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    p = model(Xt)
    loss = loss_fn(p, yt) + lam * (model.linear.weight ** 2).sum()
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0:
        print(epoch, round(loss.item(), 4))

with torch.no_grad():
    prob_nn = model(Xte).numpy()
print(pd.Series(score(y_test, prob_nn)).round(3))

0 0.6931
500 0.2371
1000 0.2368
1500 0.2368
2000 0.2368
2500 0.2368
3000 0.2368
3500 0.2368
4000 0.2368
4500 0.2368
5000 0.2368
5500 0.2368
6000 0.2368
6500 0.2368
7000 0.2368
7500 0.2368
8000 0.2368
8500 0.2368
9000 0.2368
9500 0.2368
accuracy     0.900
precision    0.774
recall       0.533
roc_auc      0.909
dtype: float64


In [18]:
results = pd.DataFrame({
    "baseline": score(y_test, prob_base),
    "sklearn": score(y_test, prob_sk),
    "manual": score(y_test, prob_manual),
    "nn.Module": score(y_test, prob_nn),
}).T.round(4)
print(results)

weights_model1 = classifier.coef_.ravel()
weights_model2 = w.detach().numpy()
weights_model3 = model.linear.weight.detach().numpy().ravel()

print("max weight gap, model1 vs model2:", np.abs(weights_model1 - weights_model2).max().round(4))
print("max weight gap, model1 vs model3:", np.abs(weights_model1 - weights_model3).max().round(4))
print("max weight gap, model2 vs model3:", np.abs(weights_model2 - weights_model3).max().round(6))
print("max probability gap, model2 vs model3:", np.abs(prob_manual - prob_nn).max().round(6))

           accuracy  precision  recall  roc_auc
baseline     0.8395     0.0000  0.0000   0.5000
sklearn      0.9000     0.7738  0.5328   0.9093
manual       0.9000     0.7738  0.5328   0.9093
nn.Module    0.9000     0.7738  0.5328   0.9093
max weight gap, model1 vs model2: 0.008
max weight gap, model1 vs model3: 0.008
max weight gap, model2 vs model3: 0.0
max probability gap, model2 vs model3: 0.0


In [19]:
weights_table = pd.DataFrame({
    "sklearn": weights_model1,
    "manual": weights_model2,
    "nn.Module": weights_model3,
}, index=X.columns)
weights_table["gap"] = (weights_table["sklearn"] - weights_table["manual"]).abs()

ranked = weights_table["sklearn"].sort_values(key=abs, ascending=False)
display(ranked.round(2))

Total_Trans_Ct                   -2.54
Total_Trans_Amt                   1.40
Total_Relationship_Count         -0.77
Total_Revolving_Bal              -0.73
Total_Ct_Chng_Q4_Q1              -0.68
Contacts_Count_12_mon             0.52
Months_Inactive_12_mon            0.52
Income_Category_$60K - $80K      -0.17
Income_Category_Less than $40K    0.14
Months_on_book                   -0.09
Income_Category_$40K - $60K      -0.09
Avg_Utilization_Ratio            -0.08
Income_Category_$80K - $120K     -0.08
Income_Category_Unknown           0.07
Credit_Limit                     -0.03
Customer_Age                     -0.02
Name: sklearn, dtype: float64

In [21]:
Path("models").mkdir(exist_ok=True)

joblib.dump({"scaler": scaler, "features": list(X.columns)}, "models/preprocessing.joblib")
joblib.dump(classifier, "models/sklearn_model.joblib")
torch.save({"w": w.detach(), "b": b.detach()}, "models/manual_model.pt")
torch.save(model.state_dict(), "models/nn_model.pt")

pd.DataFrame({
    "actual": y_test.to_numpy(),
    "baseline": prob_base,
    "sklearn": prob_sk,
    "manual": prob_manual,
    "nn_module": prob_nn,
}).to_csv("models/test_predictions.csv", index=False)

df.to_csv("models/clean_data.csv", index=False)
results.to_csv("models/comparison_table.csv")
weights_table.to_csv("models/weights_table.csv")

print(sorted(p.name for p in Path("models").iterdir()))

['clean_data.csv', 'comparison_table.csv', 'manual_model.pt', 'nn_model.pt', 'preprocessing.joblib', 'sklearn_model.joblib', 'test_predictions.csv', 'weights_table.csv']
